# VORTEX-HRM Benchmark on Google Colab

Runs 50-QA benchmark with auto-push results to GitHub.

**Workflow:**
1. Mount Drive (checkpoint persistence if session drops)
2. Clone repo
3. Install ollama + pull `qwen2.5:7b`
4. Run `batch_runner.py` — checkpoints after every question
5. Copy results + config → `notebooks/colab_runs/<timestamp>/`
6. Git commit + push back to GitHub → я вижу результаты и анализирую

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dizel0110/Text-HRM-RAG/blob/main/notebooks/vortex_benchmark_colab.ipynb)

### 0. Setup GitHub token

**Один раз:**
1. https://github.com/settings/tokens → Generate new token (classic) → repo ✓ → Copy
2. В Colab слева 🔑 **Secrets** → Add secret → имя `GITHUB_TOKEN`, вставьте токен

Ноутбук сам прочитает `userdata`.

In [ ]:
import os, sys, json, time, datetime, shutil, glob, subprocess, copy
from getpass import getpass

# Try Colab userdata first, else prompt
GITHUB_TOKEN = None
try:
    from google.colab import userdata
    GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
except Exception:
    pass

if not GITHUB_TOKEN:
    GITHUB_TOKEN = getpass("Enter GitHub token (ghp_...): ").strip()

print("GitHub token set:", GITHUB_TOKEN[:8] + "..." if GITHUB_TOKEN else "MISSING")

### 0.5 Choose mode

- `"test"` → 5 вопросов, в `colab_runs/_test/`
- `"full"` → 50 вопросов, в `colab_runs/`
- `RUN_TYPE = "vortex"` → VORTEX-HRM (планировщик + исполнитель)
- `RUN_TYPE = "baseline"` → Naive RAG (один retrieve + ответ)

In [ ]:
MODE = "full"      # "test" (5 Q) или "full" (50 Q)
RUN_TYPE = "baseline"  # "vortex" (VORTEX-HRM) или "baseline" (Naive RAG)

if MODE == "test":
    NUM_QUESTIONS = 5
    RUN_PREFIX = f"{RUN_TYPE}/_test"
    print(f"MODE: TEST — {NUM_QUESTIONS} вопросов, быстрый прогон")
else:
    NUM_QUESTIONS = 50
    RUN_PREFIX = RUN_TYPE
    print(f"MODE: FULL — {NUM_QUESTIONS} вопросов, полный бенчмарк")
print(f"RUN_TYPE: {RUN_TYPE}")

### 1. Mount Google Drive

In [ ]:
from google.colab import drive
DRIVE_DIR = "/content/drive/MyDrive/vortex-hrm-benchmark"
drive.mount("/content/drive")
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f"Drive: {DRIVE_DIR}")

### 2. Clone repository

In [ ]:
REPO = "dizel0110/Text-HRM-RAG"
REPO_DIR = f"/content/{REPO.split('/')[-1]}"

# Clone with token so we can push later
CLONE_URL = f"https://{GITHUB_TOKEN}@github.com/{REPO}.git"
if not os.path.exists(REPO_DIR):
    !git clone {CLONE_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull {CLONE_URL}

!pip install -r {os.path.join(REPO_DIR, 'vortex-hrm', 'requirements.txt')} -q
print("Repo ready")

### 3. Install ollama + pull model

In [ ]:
!apt-get install -y zstd lshw -qq
!curl -fsSL https://ollama.com/install.sh | sh
proc = subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)
!ollama pull qwen2.5:7b
print("")
gpu_check = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True)
HAS_GPU = bool(gpu_check.stdout.strip())
if HAS_GPU:
    print(f"✅ GPU detected: {gpu_check.stdout.strip()}")
else:
    print("⚠️  No GPU detected — ollama will run on CPU (slow)")

if MODE == "full" and not HAS_GPU:
    raise RuntimeError("Full mode requires GPU. Set MODE = 'test' for CPU, or switch to a GPU runtime.")

print("ollama + model ready")

### 3.5 Prepare questions

Full run → все 50 вопросов. Test run → первые 5.

In [ ]:
VORTEX_DIR = os.path.join(REPO_DIR, "vortex-hrm")
QUESTIONS_SRC = os.path.join(VORTEX_DIR, "data", "multi_domain", "questions.json")

with open(QUESTIONS_SRC, encoding="utf-8") as f:
    all_qa = json.load(f)

print(f"Loaded {len(all_qa)} questions total")

if MODE == "test":
    test_qa = all_qa[:NUM_QUESTIONS]
    test_path = f"/tmp/questions_test.json"
    with open(test_path, "w", encoding="utf-8") as f:
        json.dump(test_qa, f, ensure_ascii=False, indent=2)
    QUESTIONS_FILE = test_path
    print(f"Test mode: using first {NUM_QUESTIONS} questions → {test_path}")
else:
    QUESTIONS_FILE = QUESTIONS_SRC
    print(f"Full mode: all {len(all_qa)} questions")

### 4. Show checkpoint status (resume if session dropped)

In [ ]:
ckpt_subdir = f"{RUN_TYPE}/_test" if MODE == "test" else RUN_TYPE
drive_output_dir = os.path.join(DRIVE_DIR, ckpt_subdir)
os.makedirs(drive_output_dir, exist_ok=True)

# Clear old checkpoint — fresh start every run
for fname in ["checkpoint.json", "predictions.jsonl", "errors.log"]:
    p = os.path.join(drive_output_dir, fname)
    if os.path.exists(p):
        os.remove(p)
print(f"Fresh start: 0/{NUM_QUESTIONS} → {drive_output_dir}")

### 5. Run benchmark

In [ ]:
CONFIG_NAME = "colab-gpu.yaml" if MODE == "full" else "local.yaml"
CONFIG_FILE = os.path.join(VORTEX_DIR, "configs", CONFIG_NAME)
CORPUS_FILE = os.path.join(VORTEX_DIR, "data", "multi_domain", "corpus.json")

if RUN_TYPE == "baseline":
    SCRIPT = os.path.join(VORTEX_DIR, "scripts", "baseline_rag.py")
else:
    SCRIPT = os.path.join(VORTEX_DIR, "scripts", "batch_runner.py")

print(f"Config: {CONFIG_NAME}   Run: {RUN_TYPE}")
!python {SCRIPT} \
    --config {CONFIG_FILE} \
    --questions {QUESTIONS_FILE} \
    --corpus {CORPUS_FILE} \
    --output {drive_output_dir}
print("\nBenchmark done.")

### 6. Copy results to repo and push to GitHub

In [ ]:
RUN_ID = datetime.datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
COLAB_RUNS = os.path.join(REPO_DIR, "notebooks", "colab_runs", RUN_PREFIX, RUN_ID)
os.makedirs(COLAB_RUNS, exist_ok=True)

for fname in ["predictions.jsonl", "checkpoint.json", "errors.log"]:
    src = os.path.join(drive_output_dir, fname)
    if os.path.exists(src):
        shutil.copy2(src, os.path.join(COLAB_RUNS, fname))

meta = {
    "mode": MODE,
    "run_id": RUN_ID,
    "config": CONFIG_NAME,
    "model": "qwen2.5:7b",
    "num_questions": NUM_QUESTIONS,
    "questions": QUESTIONS_FILE,
    "corpus": "data/multi_domain/corpus.json",
    "engine": "VortexEngine",
    "run_type": RUN_TYPE,
    "colab_gpu": "T4",
}
with open(os.path.join(COLAB_RUNS, "meta.json"), "w") as f:
    json.dump(meta, f, indent=2)

print(f"Copied results → {COLAB_RUNS}")

In [ ]:
!git -C {REPO_DIR} config user.email "colab@vortex-hrm.dev"
!git -C {REPO_DIR} config user.name "Colab Runner"
!git -C {REPO_DIR} remote set-url origin {CLONE_URL}

run_path = os.path.join("notebooks/colab_runs", RUN_PREFIX, RUN_ID).replace("\\", "/")
!git -C {REPO_DIR} add {run_path}/
commit_msg = f"colab {'test' if MODE == 'test' else RUN_TYPE} {RUN_ID}"
!git -C {REPO_DIR} commit -m "{commit_msg}"
result = !git -C {REPO_DIR} push origin main 2>&1
for line in result:
    print(line)

print("\n✅ Results pushed to GitHub!")
print(f"   notebooks/colab_runs/{RUN_PREFIX}/{RUN_ID}/")

### 7. Evaluate results

In [ ]:
EVAL_SCRIPT = os.path.join(VORTEX_DIR, "scripts", "eval.py")
!python {EVAL_SCRIPT} --predictions {drive_output_dir}/predictions.jsonl

---

**После пуша** напиши мне: "результат в колабе" — я посмотрю метрики и предложу следующий эксперимент.